# Local Helioseismology — Implementation / 국소 일진학 — 구현

**Paper / 논문**: Gizon, L. & Birch, A. C. (2005). "Local Helioseismology." *Living Reviews in Solar Physics*, 2, 6. [DOI: 10.12942/lrsp-2005-6](https://doi.org/10.12942/lrsp-2005-6)

---

이 notebook은 Gizon & Birch (2005)의 국소 일진학(Local Helioseismology) 핵심 개념을 수치적으로 구현합니다.
각 Part는 논문의 주요 분석 기법을 재현하며, 태양 내부 구조와 유동(flow) 측정의 원리를 시각적으로 보여줍니다.

This notebook implements key concepts from Gizon & Birch (2005) on Local Helioseismology.
Each Part reproduces a major analysis technique from the paper, visually demonstrating the principles behind probing solar interior structure and flows.

---

## 목차 / Table of Contents

| Part | 제목 / Title | 관련 논문 섹션 / Paper Section |
|------|-------------|-------------------------------|
| 1 | 태양 진동 Power Spectrum (l-ν diagram) / Solar Oscillation Power Spectrum | §2 — Solar Oscillations |
| 2 | Ring-Diagram 분석 시뮬레이션 / Ring-Diagram Analysis Simulation | §4 — Ring-Diagram Analysis |
| 3 | Time-Distance Cross-Covariance / 시간-거리 교차공분산 | §5 — Time-Distance Helioseismology |
| 4 | Travel Time 및 유동 검출 / Travel Time & Flow Detection | §5.2–5.3 — Travel-Time Measurements |
| 5 | 민감도 커널: Ray vs Fresnel Zone / Sensitivity Kernels | §6 — Sensitivity Kernels |

In [ ]:
import numpy as np
from scipy import signal, optimize, special
import matplotlib.pyplot as plt
from matplotlib import cm

# Global plot settings for publication-quality figures.
plt.rcParams.update({
    "figure.figsize": (10, 7),
    "figure.dpi": 120,
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "image.cmap": "inferno",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Solar constants.
R_SUN = 6.96e8       # Solar radius [m]
G_SURF = 274.0       # Surface gravity [m/s^2]
C_AVG = 1.5e5        # Average sound speed [m/s] (rough estimate for interior)

print("Imports and constants loaded successfully.")

---
## Part 1: 태양 진동 Power Spectrum (l-ν diagram) / Solar Oscillation Power Spectrum

### 배경 / Background

태양은 수백만 개의 음향 모드(acoustic modes, p-modes)와 표면 중력 모드(f-mode)로 진동합니다.
이 진동은 **spherical harmonics** $Y_l^m(\theta, \phi)$로 분해되며, 각 모드는 **구면 조화 차수** $l$과
**방사 차수** $n$으로 특성화됩니다. $l$-$\nu$ diagram은 이러한 모드들의 분산 관계(dispersion relation)를
보여주는 핵심 진단 도구입니다.

The Sun oscillates in millions of acoustic modes (p-modes) and surface gravity modes (f-mode).
These oscillations are decomposed into **spherical harmonics** $Y_l^m(\theta, \phi)$, and each mode
is characterized by the **spherical harmonic degree** $l$ and **radial order** $n$. The $l$-$\nu$ diagram
is a key diagnostic tool that displays the dispersion relations of these modes.

### 분산 관계 / Dispersion Relations

**f-mode (표면 중력 모드 / surface gravity mode)**:
$$\nu_f = \frac{1}{2\pi}\sqrt{\frac{g \cdot l}{R_\odot}}$$

여기서 $g = 274$ m/s²는 태양 표면 중력, $R_\odot = 6.96 \times 10^8$ m는 태양 반지름입니다.

where $g = 274$ m/s² is the solar surface gravity and $R_\odot = 6.96 \times 10^8$ m is the solar radius.

**p-modes (음향 모드 / acoustic modes)**:

Duvall의 법칙에 기반한 근사 분산 관계를 사용합니다:

We use an approximate dispersion relation based on Duvall's law:

$$\nu_{n,l} \approx \left(n + \frac{l}{2} + \alpha\right) \cdot \Delta\nu$$

여기서 $\Delta\nu \approx 135\;\mu\text{Hz}$는 large frequency separation, $\alpha \approx 1.5$는 위상 보정 상수입니다.
보다 정확한 모델링을 위해 2차 보정항(second-order correction)도 추가합니다.

where $\Delta\nu \approx 135\;\mu\text{Hz}$ is the large frequency separation and $\alpha \approx 1.5$ is a phase correction constant. We also add a second-order correction for more accurate modeling.

In [ ]:
# =============================================================================
# Part 1: Solar Oscillation Power Spectrum (l-nu diagram)
# =============================================================================

# Spherical harmonic degree range.
l_arr = np.arange(0, 1501)

# --- f-mode dispersion relation ---
# nu_f = (1/2pi) * sqrt(g * l / R_sun)
nu_f = (1.0 / (2.0 * np.pi)) * np.sqrt(G_SURF * l_arr / R_SUN)
nu_f_mhz = nu_f * 1e3  # Convert Hz to mHz.

# --- p-mode dispersion relation (Duvall-law approximation) ---
# nu_{n,l} ~ (n + l/2 + alpha) * Delta_nu
# with a second-order curvature correction to bend ridges realistically.
DELTA_NU = 135e-6   # Large frequency separation [Hz] (~135 muHz).
ALPHA = 1.5          # Phase correction constant.
D0 = 1.5e-6          # Small separation parameter [Hz] for curvature.

n_radial = np.arange(1, 26)  # Radial orders n = 1..25.

fig, ax = plt.subplots(figsize=(12, 8))

# Plot p-mode ridges.
for n in n_radial:
    # Base asymptotic frequency.
    nu_nl = (n + l_arr / 2.0 + ALPHA) * DELTA_NU

    # Second-order correction: bends high-l portion downward.
    # This mimics the acoustic cutoff and turning-point effects.
    correction = -D0 * l_arr * (l_arr + 1) / (n + l_arr / 2.0 + ALPHA)
    nu_nl = nu_nl + correction

    nu_nl_mhz = nu_nl * 1e3  # Convert to mHz.

    # Only plot modes within observable range (0 < nu < 6 mHz).
    mask = (nu_nl_mhz > 0) & (nu_nl_mhz < 6.0)
    ax.plot(l_arr[mask], nu_nl_mhz[mask], "b-", lw=0.6, alpha=0.8)

# Plot f-mode.
mask_f = (nu_f_mhz > 0) & (nu_f_mhz < 6.0) & (l_arr > 0)
ax.plot(l_arr[mask_f], nu_f_mhz[mask_f], "r-", lw=1.5, label="f-mode")

# Annotations.
ax.set_xlabel(r"Spherical Harmonic Degree $\ell$")
ax.set_ylabel(r"Frequency $\nu$ [mHz]")
ax.set_title(r"$\ell$–$\nu$ Diagram (Solar Oscillation Power Spectrum)")
ax.set_xlim(0, 1500)
ax.set_ylim(0, 6)
ax.legend(loc="upper left", fontsize=12)

# Label a few ridges.
for n_label in [1, 5, 10, 15, 20, 25]:
    l_pos = 200
    nu_label = (n_label + l_pos / 2.0 + ALPHA) * DELTA_NU * 1e3
    corr_label = -D0 * l_pos * (l_pos + 1) / (n_label + l_pos / 2.0 + ALPHA) * 1e3
    nu_label += corr_label
    if 0 < nu_label < 6.0:
        ax.annotate(
            f"n={n_label}", (l_pos, nu_label),
            fontsize=8, color="navy",
            xytext=(5, 5), textcoords="offset points",
        )

plt.tight_layout()
plt.show()

---
## Part 2: Ring-Diagram 분석 시뮬레이션 / Ring-Diagram Analysis Simulation

### 배경 / Background

Ring-diagram 분석은 국소 일진학의 핵심 기법 중 하나입니다 (논문 §4).
태양 표면의 제한된 영역(patch)에서 관측된 진동 데이터를 3D Fourier 변환하면
power spectrum $P(k_x, k_y, \omega)$를 얻습니다. 일정한 주파수 $\omega$에서의
$k_x$-$k_y$ 단면은 **동심원(ring)** 형태를 보입니다 — 이것이 "ring diagram"입니다.

Ring-diagram analysis is one of the key techniques in local helioseismology (Paper §4).
When oscillation data from a limited patch of the solar surface is 3D Fourier-transformed,
we obtain the power spectrum $P(k_x, k_y, \omega)$. Cross-sections at constant frequency $\omega$
in the $k_x$-$k_y$ plane show **concentric rings** — hence "ring diagrams."

### 유동에 의한 ring 이동 / Ring Shift Due to Flows

수평 유동 $\mathbf{U} = (U_x, U_y)$이 존재하면, 분산 관계가 Doppler 이동됩니다:

When a horizontal flow $\mathbf{U} = (U_x, U_y)$ is present, the dispersion relation is Doppler-shifted:

$$\omega \rightarrow \omega - \mathbf{k} \cdot \mathbf{U} = \omega - k_x U_x - k_y U_y$$

이로 인해 ring이 $(k_x, k_y)$ 평면에서 **이동(shift)**합니다.
이동량을 측정하면 수평 유동 속도를 추정할 수 있습니다.

This causes the rings to **shift** in the $(k_x, k_y)$ plane.
By measuring the shift, we can estimate the horizontal flow velocity.

### 구현 내용 / Implementation

1. Lorentzian ridge를 가진 합성 3D power spectrum 생성
2. 유동이 없는 경우와 있는 경우의 ring 비교
3. Ring fitting으로 유동 속도 복원

In [ ]:
# =============================================================================
# Part 2: Ring-Diagram Analysis Simulation
# =============================================================================

def make_ring_power(kx_grid, ky_grid, omega_target, k_ring, gamma, ux=0.0, uy=0.0):
    """Create a 2D power spectrum slice at constant omega with Lorentzian ridges.

    Args:
        kx_grid: 2D array of kx values [rad/Mm].
        ky_grid: 2D array of ky values [rad/Mm].
        omega_target: Target angular frequency [rad/s].
        k_ring: Wavenumber of the ring (mode k at this omega) [rad/Mm].
        gamma: Half-width at half-maximum of the Lorentzian [rad/s].
        ux: Horizontal flow in x direction [m/s].
        uy: Horizontal flow in y direction [m/s].

    Returns:
        2D array of power spectrum values.
    """
    k_mag = np.sqrt(kx_grid**2 + ky_grid**2)

    # Convert flow from m/s to Mm/s for consistency with k in rad/Mm.
    ux_mm = ux * 1e-6  # m/s -> Mm/s
    uy_mm = uy * 1e-6

    # Doppler shift: omega_eff = omega_target - k . U
    omega_shift = kx_grid * ux_mm + ky_grid * uy_mm

    # Effective omega for the mode at each (kx, ky):
    # The mode resonance is at k = k_ring (no flow).
    # With flow, the resonance condition becomes:
    # omega_target = omega_0(k) + k . U => omega_0(k) = omega_target - k . U
    # For a given omega_target, the ring is at |k| such that omega_0(|k|) = omega_target - k.U
    # In a Lorentzian model: P ~ 1 / ((k_mag - k_eff)^2 + (gamma_k)^2)
    # where k_eff accounts for the flow shift.

    # Simpler approach: express as Lorentzian in k-space.
    # The resonant k shifts due to flow.
    # dw/dk ~ group velocity v_g. Then dk = dw / v_g = (k.U) / v_g.
    v_group = 1e-3  # Group velocity [Mm/s] ~ 1 km/s typical for p-modes.
    k_shift = omega_shift / v_group if v_group > 0 else 0.0
    k_eff = k_ring + k_shift

    # Lorentzian in wavenumber space.
    gamma_k = gamma / v_group  # Convert frequency width to k width.
    power = 1.0 / ((k_mag - k_eff)**2 + gamma_k**2)
    return power


# --- Set up wavenumber grid ---
nk = 256
k_max = 2.0  # [rad/Mm]
kx_1d = np.linspace(-k_max, k_max, nk)
ky_1d = np.linspace(-k_max, k_max, nk)
kx_grid, ky_grid = np.meshgrid(kx_1d, ky_1d)

# Ring parameters: mode at approximately 3 mHz, k_ring ~ 0.8 rad/Mm.
omega_target = 2 * np.pi * 3e-3  # 3 mHz in rad/s.
k_ring = 0.8         # [rad/Mm]
gamma = 2e-4          # Line width [rad/s].

# --- Case 1: No flow ---
power_no_flow = make_ring_power(kx_grid, ky_grid, omega_target, k_ring, gamma,
                                ux=0.0, uy=0.0)

# --- Case 2: With flow Ux=300 m/s, Uy=150 m/s ---
UX_FLOW = 300.0   # [m/s]
UY_FLOW = 150.0   # [m/s]
power_with_flow = make_ring_power(kx_grid, ky_grid, omega_target, k_ring, gamma,
                                  ux=UX_FLOW, uy=UY_FLOW)

# --- Fit the shifted ring to recover flow ---
# Find the peak radius along different azimuthal angles.
theta_fit = np.linspace(0, 2 * np.pi, 360, endpoint=False)
k_peak_no_flow = np.zeros_like(theta_fit)
k_peak_with_flow = np.zeros_like(theta_fit)

k_radial = np.linspace(0.3, 1.3, 500)

for i, theta in enumerate(theta_fit):
    kx_ray = k_radial * np.cos(theta)
    ky_ray = k_radial * np.sin(theta)

    # Interpolate power along the ray (no flow).
    from scipy.interpolate import RegularGridInterpolator
    interp_nf = RegularGridInterpolator((ky_1d, kx_1d), power_no_flow,
                                         bounds_error=False, fill_value=0)
    pts = np.column_stack([ky_ray, kx_ray])
    p_ray_nf = interp_nf(pts)
    k_peak_no_flow[i] = k_radial[np.argmax(p_ray_nf)]

    # With flow.
    interp_wf = RegularGridInterpolator((ky_1d, kx_1d), power_with_flow,
                                         bounds_error=False, fill_value=0)
    p_ray_wf = interp_wf(pts)
    k_peak_with_flow[i] = k_radial[np.argmax(p_ray_wf)]

# The peak-radius modulation: k_peak(theta) = k0 + dk_x*cos(theta) + dk_y*sin(theta)
# Fit for dk_x and dk_y.
A_fit = np.column_stack([np.ones_like(theta_fit),
                         np.cos(theta_fit),
                         np.sin(theta_fit)])
coeffs, _, _, _ = np.linalg.lstsq(A_fit, k_peak_with_flow, rcond=None)
k0_fit, dkx_fit, dky_fit = coeffs

# Convert dk back to velocity: U = dk * v_group / k (approximately).
v_group_fit = 1e-3  # Mm/s
ux_recovered = dkx_fit * v_group_fit * 1e6  # Back to m/s.
uy_recovered = dky_fit * v_group_fit * 1e6

# --- Plot results ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) Ring without flow.
ax = axes[0]
im = ax.pcolormesh(kx_1d, ky_1d, np.log10(power_no_flow + 1),
                   shading="auto", cmap="hot")
ax.set_xlabel(r"$k_x$ [rad/Mm]")
ax.set_ylabel(r"$k_y$ [rad/Mm]")
ax.set_title("Ring — No Flow\n유동 없음")
ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label=r"$\log_{10}(P+1)$")

# (2) Ring with flow.
ax = axes[1]
im = ax.pcolormesh(kx_1d, ky_1d, np.log10(power_with_flow + 1),
                   shading="auto", cmap="hot")
ax.set_xlabel(r"$k_x$ [rad/Mm]")
ax.set_ylabel(r"$k_y$ [rad/Mm]")
ax.set_title(f"Ring — With Flow ($U_x$={UX_FLOW}, $U_y$={UY_FLOW} m/s)\n유동 있음")
ax.set_aspect("equal")
plt.colorbar(im, ax=ax, label=r"$\log_{10}(P+1)$")

# (3) Fitted ring peak vs angle.
ax = axes[2]
ax.plot(np.degrees(theta_fit), k_peak_no_flow, "b-", lw=1.2, label="No flow")
ax.plot(np.degrees(theta_fit), k_peak_with_flow, "r-", lw=1.2, label="With flow")
ax.plot(np.degrees(theta_fit),
        coeffs[0] + coeffs[1]*np.cos(theta_fit) + coeffs[2]*np.sin(theta_fit),
        "k--", lw=1.5, label="Fit")
ax.set_xlabel(r"Azimuthal angle $\theta$ [deg]")
ax.set_ylabel(r"Peak $k$ [rad/Mm]")
ax.set_title("Ring Peak vs Angle / Ring 피크 vs 방위각")
ax.legend()

plt.tight_layout()
plt.show()

print(f"Input flow  / 입력 유동: Ux = {UX_FLOW:.1f} m/s, Uy = {UY_FLOW:.1f} m/s")
print(f"Recovered   / 복원 유동: Ux = {ux_recovered:.1f} m/s, Uy = {uy_recovered:.1f} m/s")

---
## Part 3: Time-Distance Cross-Covariance / 시간-거리 교차공분산

### 배경 / Background

Time-distance helioseismology (논문 §5)는 태양 표면의 서로 다른 두 지점 사이의
파동 전파 시간(travel time)을 측정하는 기법입니다. 핵심 관측량은 **교차공분산 함수
(cross-covariance function)**입니다:

Time-distance helioseismology (Paper §5) measures the wave travel time between
two different points on the solar surface. The key observable is the **cross-covariance
function**:

$$C(x_1, x_2, t) = \frac{1}{T} \int_0^T \Phi(x_1, t') \, \Phi(x_2, t' + t) \, dt'$$

여기서 $\Phi(x, t)$는 표면 파동장(wave field)입니다.

where $\Phi(x, t)$ is the surface wave field.

### 시간-거리 diagram / Time-Distance Diagram

교차공분산을 거리 $\Delta = |x_2 - x_1|$와 시간 지연 $t$의 함수로 표시하면
**time-distance diagram**을 얻습니다. 이 diagram에서는:

Plotting the cross-covariance as a function of distance $\Delta = |x_2 - x_1|$ and
time lag $t$ gives the **time-distance diagram**. In this diagram:

- **양의 시간 지연(positive lag)**: $x_1 \to x_2$로 전파하는 wave packet
- **음의 시간 지연(negative lag)**: $x_2 \to x_1$로 전파하는 wave packet
- **Positive lag**: Wave packet propagating from $x_1 \to x_2$
- **Negative lag**: Wave packet propagating from $x_2 \to x_1$

### 구현 / Implementation

모드 중첩(superposition of modes)으로 1D 파동장을 시뮬레이션합니다:

We simulate a 1D wave field using superposition of modes:

$$\Phi(x, t) = \sum_k A_k \cos(kx - \omega(k)t + \phi_k)$$

여기서 $\phi_k$는 랜덤 위상이고, $\omega(k)$는 분산 관계입니다.

where $\phi_k$ are random phases and $\omega(k)$ is the dispersion relation.

In [ ]:
# =============================================================================
# Part 3: Time-Distance Cross-Covariance
# =============================================================================

def simulate_wave_field(x_arr, t_arr, k_modes, omega_func, amplitudes=None,
                        flow_speed=0.0, rng_seed=42):
    """Simulate a 1D wave field as superposition of modes with random phases.

    Args:
        x_arr: 1D array of spatial positions [Mm].
        t_arr: 1D array of time values [s].
        k_modes: 1D array of wavenumbers [rad/Mm].
        omega_func: Function omega(k) returning angular frequency [rad/s].
        amplitudes: Optional 1D array of mode amplitudes. If None, use Gaussian envelope.
        flow_speed: Uniform horizontal flow speed [Mm/s] (Doppler shift).
        rng_seed: Random seed for reproducibility.

    Returns:
        2D array Phi(x, t) with shape (len(x_arr), len(t_arr)).
    """
    rng = np.random.default_rng(rng_seed)
    phases = rng.uniform(0, 2 * np.pi, size=len(k_modes))

    if amplitudes is None:
        # Gaussian amplitude envelope peaked around k ~ 0.5 rad/Mm.
        k_center = 0.5
        k_width = 0.3
        amplitudes = np.exp(-0.5 * ((k_modes - k_center) / k_width) ** 2)

    # Build wave field: Phi(x, t) = sum_k A_k cos(kx - omega(k)t + phi_k)
    # Use broadcasting: x_arr[:, None, None] * k_modes[None, None, :] etc.
    # But that might be large. Use loop over modes for memory efficiency.
    phi = np.zeros((len(x_arr), len(t_arr)))
    for i, k in enumerate(k_modes):
        omega_k = omega_func(k) + k * flow_speed  # Doppler shift.
        phi += amplitudes[i] * np.cos(
            k * x_arr[:, None] - omega_k * t_arr[None, :] + phases[i]
        )
    return phi


def compute_cross_covariance(phi, x_arr, t_arr, x_ref_idx=0):
    """Compute cross-covariance C(Delta, tau) relative to a reference point.

    Args:
        phi: 2D wave field array (n_x, n_t).
        x_arr: 1D spatial coordinate array [Mm].
        t_arr: 1D time array [s].
        x_ref_idx: Index of the reference point x1.

    Returns:
        Tuple of (delta_arr, tau_arr, C) where C is 2D cross-covariance.
    """
    n_x, n_t = phi.shape
    ref_signal = phi[x_ref_idx, :]  # Phi(x1, t')

    # Use FFT-based cross-correlation for efficiency.
    # C(Delta, tau) = (1/T) * sum_t' Phi(x1, t') * Phi(x2, t' + tau)
    ref_fft = np.fft.rfft(ref_signal)
    n_tau = n_t
    tau_arr = (np.arange(n_tau) - n_tau // 2) * (t_arr[1] - t_arr[0])

    C = np.zeros((n_x, n_tau))
    for ix in range(n_x):
        sig_fft = np.fft.rfft(phi[ix, :])
        # Cross-correlation via FFT: IFFT(conj(F1) * F2).
        cc = np.fft.irfft(np.conj(ref_fft) * sig_fft, n=n_t)
        # Shift so zero lag is in the center.
        C[ix, :] = np.fft.fftshift(cc) / n_t

    delta_arr = x_arr - x_arr[x_ref_idx]
    return delta_arr, tau_arr, C


# --- Parameters ---
# Spatial grid: 0 to 200 Mm.
nx = 200
x_arr = np.linspace(0, 200, nx)  # [Mm]

# Time grid: 0 to 20000 s (~5.5 hours) with 60 s cadence.
dt = 60.0  # [s]
nt = 2048
t_arr = np.arange(nt) * dt  # [s]

# Wavenumber modes: positive k only (will give symmetric covariance).
n_modes = 150
k_modes = np.linspace(0.05, 2.0, n_modes)  # [rad/Mm]

# Dispersion relation: omega(k) ~ c_s * k with slight dispersion.
# Use acoustic-like dispersion: omega = sqrt(c_s^2 * k^2 + omega_cutoff^2)
# c_s ~ 7 km/s = 0.007 Mm/s, omega_cutoff ~ 2*pi*5.3 mHz (acoustic cutoff).
C_S = 0.007       # Sound speed [Mm/s] (typical photospheric).
OMEGA_CUT = 2 * np.pi * 5.3e-3  # Acoustic cutoff [rad/s].


def omega_acoustic(k):
    """Acoustic dispersion relation with cutoff frequency.

    Args:
        k: Wavenumber [rad/Mm].

    Returns:
        Angular frequency [rad/s].
    """
    return np.sqrt(C_S**2 * k**2 + OMEGA_CUT**2)


# --- Simulate wave field (no flow) ---
print("Simulating wave field (no flow)...")
phi_no_flow = simulate_wave_field(x_arr, t_arr, k_modes, omega_acoustic,
                                   flow_speed=0.0, rng_seed=42)

# --- Compute cross-covariance ---
# Reference point near center of domain.
x_ref_idx = nx // 4
print("Computing cross-covariance...")
delta_arr, tau_arr, C_cc = compute_cross_covariance(phi_no_flow, x_arr, t_arr,
                                                     x_ref_idx=x_ref_idx)

# --- Plot time-distance diagram ---
fig, ax = plt.subplots(figsize=(12, 8))

# Only show positive distances and reasonable time lags.
tau_min, tau_max = -8000, 8000  # [s]
delta_min, delta_max = 0, 150    # [Mm]

tau_mask = (tau_arr >= tau_min) & (tau_arr <= tau_max)
delta_mask = (delta_arr >= delta_min) & (delta_arr <= delta_max)

C_plot = C_cc[np.ix_(delta_mask, tau_mask)]
tau_plot = tau_arr[tau_mask]
delta_plot = delta_arr[delta_mask]

# Normalize for display.
vmax = np.percentile(np.abs(C_plot), 99)

im = ax.pcolormesh(tau_plot / 60.0, delta_plot, C_plot,
                   shading="auto", cmap="seismic", vmin=-vmax, vmax=vmax)
ax.set_xlabel("Time Lag / 시간 지연 $\\tau$ [min]")
ax.set_ylabel("Distance / 거리 $\\Delta$ [Mm]")
ax.set_title("Time-Distance Diagram (Cross-Covariance)\n시간-거리 diagram (교차공분산)")
plt.colorbar(im, ax=ax, label="$C(\\Delta, \\tau)$")

ax.axvline(0, color="white", ls="--", lw=0.8, alpha=0.7)

plt.tight_layout()
plt.show()
print("Positive-lag branch: waves propagating away from reference point.")
print("Negative-lag branch: waves propagating toward reference point.")

---
## Part 4: Travel Time 및 유동 검출 / Travel Time & Flow Detection

### 배경 / Background

유동(flow)이 존재하면 파동의 전파 시간이 비대칭이 됩니다 (논문 §5.2–5.3).
유동 방향으로 전파하는 파동은 빨라지고, 반대 방향으로 전파하는 파동은 느려집니다.

In the presence of flows, wave travel times become asymmetric (Paper §5.2–5.3).
Waves propagating in the flow direction travel faster, while waves propagating
against the flow travel slower.

### Travel-time 측정 / Travel-Time Measurements

교차공분산의 positive-lag 피크 시간 $\tau_+$와 negative-lag 피크 시간 $\tau_-$를 측정하면:

By measuring the positive-lag peak time $\tau_+$ and negative-lag peak time $\tau_-$ of the cross-covariance:

$$\tau_{\text{diff}} = \tau_+ - |\tau_-|$$

이 **differential travel time**은 유동 속도 $U$에 비례합니다:

This **differential travel time** is proportional to the flow speed $U$:

$$\tau_{\text{diff}} \propto U$$

### 구현 / Implementation

1. Part 3의 시뮬레이션에 균일 유동(uniform flow) 추가
2. 교차공분산의 비대칭성 확인
3. 여러 유동 속도에 대해 $\tau_{\text{diff}}$ 측정
4. $\tau_{\text{diff}}$ vs $U$ 의 선형 관계 확인

In [ ]:
# =============================================================================
# Part 4: Travel Time & Flow Detection
# =============================================================================

def measure_travel_times(C_cc, tau_arr, delta_idx, tau_window=(500, 5000)):
    """Measure positive and negative lag peak times from cross-covariance.

    Args:
        C_cc: 2D cross-covariance array (n_x, n_tau).
        tau_arr: 1D array of time lags [s].
        delta_idx: Spatial index for the distance at which to measure.
        tau_window: Tuple (tau_min, tau_max) in seconds to search for peaks.

    Returns:
        Tuple (tau_plus, tau_minus) — peak times for positive and negative lags [s].
    """
    cc_slice = C_cc[delta_idx, :]

    # Positive lag window.
    pos_mask = (tau_arr >= tau_window[0]) & (tau_arr <= tau_window[1])
    pos_idx = np.where(pos_mask)[0]
    if len(pos_idx) > 0:
        peak_pos = pos_idx[np.argmax(np.abs(cc_slice[pos_idx]))]
        tau_plus = tau_arr[peak_pos]
    else:
        tau_plus = np.nan

    # Negative lag window.
    neg_mask = (tau_arr >= -tau_window[1]) & (tau_arr <= -tau_window[0])
    neg_idx = np.where(neg_mask)[0]
    if len(neg_idx) > 0:
        peak_neg = neg_idx[np.argmax(np.abs(cc_slice[neg_idx]))]
        tau_minus = tau_arr[peak_neg]
    else:
        tau_minus = np.nan

    return tau_plus, tau_minus


# --- Demonstrate asymmetry with a single flow ---
print("Simulating wave field with flow U = 0.0005 Mm/s (500 m/s)...")
flow_demo = 0.0005  # Mm/s = 500 m/s
phi_flow = simulate_wave_field(x_arr, t_arr, k_modes, omega_acoustic,
                                flow_speed=flow_demo, rng_seed=42)
delta_flow, tau_flow, C_flow = compute_cross_covariance(phi_flow, x_arr, t_arr,
                                                         x_ref_idx=x_ref_idx)

# Compare cross-covariance at a specific distance.
delta_target = 30.0  # Mm
delta_target_idx = np.argmin(np.abs(delta_arr - delta_target))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) No flow — symmetric.
ax = axes[0]
tau_mask_plot = (tau_arr >= -6000) & (tau_arr <= 6000)
ax.plot(tau_arr[tau_mask_plot] / 60.0, C_cc[delta_target_idx, tau_mask_plot],
        "b-", lw=1.2)
ax.set_xlabel("Time Lag / 시간 지연 $\\tau$ [min]")
ax.set_ylabel("$C(\\Delta, \\tau)$")
ax.set_title(f"No Flow — $\\Delta$ = {delta_target:.0f} Mm\n유동 없음")
ax.axvline(0, color="gray", ls="--", lw=0.8)

# (2) With flow — asymmetric.
ax = axes[1]
ax.plot(tau_flow[tau_mask_plot] / 60.0, C_flow[delta_target_idx, tau_mask_plot],
        "r-", lw=1.2)
ax.set_xlabel("Time Lag / 시간 지연 $\\tau$ [min]")
ax.set_ylabel("$C(\\Delta, \\tau)$")
ax.set_title(f"With Flow U = {flow_demo*1e3:.0f} m/s — $\\Delta$ = {delta_target:.0f} Mm\n유동 있음")
ax.axvline(0, color="gray", ls="--", lw=0.8)

plt.tight_layout()
plt.show()

# --- Scan over multiple flow speeds ---
print("\nScanning flow speeds for tau_diff measurement...")
flow_speeds = np.linspace(-0.001, 0.001, 15)  # [Mm/s] = -1000 to 1000 m/s
tau_diffs = []

for u in flow_speeds:
    phi_u = simulate_wave_field(x_arr, t_arr, k_modes, omega_acoustic,
                                 flow_speed=u, rng_seed=42)
    _, tau_u, C_u = compute_cross_covariance(phi_u, x_arr, t_arr,
                                              x_ref_idx=x_ref_idx)
    tau_p, tau_m = measure_travel_times(C_u, tau_u, delta_target_idx,
                                         tau_window=(300, 6000))
    tau_diff = tau_p - np.abs(tau_m)
    tau_diffs.append(tau_diff)
    print(f"  U = {u*1e3:+7.1f} m/s  |  tau+ = {tau_p:8.1f} s  |  "
          f"tau- = {tau_m:8.1f} s  |  tau_diff = {tau_diff:+8.1f} s")

tau_diffs = np.array(tau_diffs)
flow_speeds_ms = flow_speeds * 1e3  # Convert to m/s.

# --- Linear fit ---
valid = ~np.isnan(tau_diffs)
coeffs_lin = np.polyfit(flow_speeds_ms[valid], tau_diffs[valid], 1)
fit_line = np.polyval(coeffs_lin, flow_speeds_ms)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(flow_speeds_ms[valid], tau_diffs[valid], "ro", ms=8, label="Measured $\\tau_{\\rm diff}$")
ax.plot(flow_speeds_ms, fit_line, "k--", lw=1.5,
        label=f"Linear fit: slope = {coeffs_lin[0]:.4f} s/(m/s)")
ax.set_xlabel("Flow Speed $U$ [m/s] / 유동 속도")
ax.set_ylabel("$\\tau_{\\rm diff} = \\tau_+ - |\\tau_-|$ [s]")
ax.set_title("Differential Travel Time vs Flow Speed\n차등 전파 시간 vs 유동 속도")
ax.legend(fontsize=12)
ax.axhline(0, color="gray", ls=":", lw=0.8)
ax.axvline(0, color="gray", ls=":", lw=0.8)

plt.tight_layout()
plt.show()

print(f"\nLinear fit: tau_diff = {coeffs_lin[0]:.4f} * U + {coeffs_lin[1]:.2f}")
print("The linear relationship confirms that differential travel time is "
      "proportional to flow speed.")

---
## Part 5: 민감도 커널 — Ray vs Fresnel Zone / Sensitivity Kernels — Ray vs Fresnel Zone

### 배경 / Background

Travel-time 측정값은 태양 내부 구조의 **적분(integral)** 측정입니다 (논문 §6).
이 적분의 가중 함수가 바로 **sensitivity kernel** $K(\mathbf{r})$입니다:

Travel-time measurements are **integral** measurements of the solar interior structure (Paper §6).
The weighting function of this integral is the **sensitivity kernel** $K(\mathbf{r})$:

$$\delta\tau = \int K(\mathbf{r}) \, \delta q(\mathbf{r}) \, d^3\mathbf{r}$$

여기서 $\delta q(\mathbf{r})$는 내부 구조(음속, 유동 등)의 섭동입니다.

where $\delta q(\mathbf{r})$ is a perturbation in interior structure (sound speed, flow, etc.).

### Ray 커널 vs Fresnel Zone 커널 / Ray Kernel vs Fresnel Zone Kernel

**Ray 근사 (geometric optics)**:
- 민감도가 ray path 위에만 집중됨: $K(\mathbf{r}) = \delta(\mathbf{r} - \text{ray path})$
- 유한 파장 효과를 무시

**Ray approximation (geometric optics)**:
- Sensitivity concentrated only on the ray path: $K(\mathbf{r}) = \delta(\mathbf{r} - \text{ray path})$
- Ignores finite-wavelength effects

**Fresnel zone 커널 (finite-wavelength)**:
- "Banana-doughnut" 형태: ray path를 따라 **속이 빈(hollow)** 구조
- Ray 위에서는 민감도가 0에 가깝고, 첫 번째 Fresnel zone에서 최대
- 논문 Eq. 65에 기반

**Fresnel zone kernel (finite-wavelength)**:
- "Banana-doughnut" shape: **hollow** structure along the ray path
- Near-zero sensitivity on the ray itself, maximum at the first Fresnel zone
- Based on Paper Eq. 65

### 음속 프로파일 / Sound Speed Profile

단순화된 음속 프로파일을 사용합니다:

We use a simplified sound speed profile:

$$c(z) = c_0 \sqrt{1 + z/H}$$

여기서 $c_0$는 표면 음속, $H$는 scale height, $z$는 깊이(아래 방향 양수)입니다.

where $c_0$ is the surface sound speed, $H$ is the scale height, and $z$ is depth (positive downward).

In [ ]:
# =============================================================================
# Part 5: Sensitivity Kernels — Ray vs Fresnel Zone
# =============================================================================

# --- Sound speed profile ---
C0 = 7.0    # Surface sound speed [km/s] = 0.007 Mm/s.
H = 50.0    # Scale height [Mm].


def sound_speed(z):
    """Sound speed as a function of depth.

    Args:
        z: Depth below surface [Mm] (positive downward).

    Returns:
        Sound speed [Mm/s].
    """
    return (C0 * 1e-3) * np.sqrt(1 + z / H)  # [Mm/s]


def compute_ray_path(x_sep, z_max=None, n_steps=500):
    """Compute an acoustic ray path between two surface points.

    Uses the analytic solution for c(z) = c0*sqrt(1+z/H).
    The ray is parameterized as an arc: the turning depth z_t satisfies
    c(z_t)/sin(theta_t) = c(0)/sin(theta_0), where theta_t = pi/2 at turning point.

    For c(z) = c0*sqrt(1+z/H), rays are circular arcs.

    Args:
        x_sep: Horizontal separation between source and receiver [Mm].
        z_max: Optional maximum depth to limit ray [Mm].
        n_steps: Number of points along the ray.

    Returns:
        Tuple (x_ray, z_ray, travel_time) — arrays of ray coordinates [Mm]
        and total travel time [s].
    """
    c0 = C0 * 1e-3  # [Mm/s]

    # For c(z) = c0*sqrt(1+z/H), rays are arcs of circles.
    # The turning depth z_t and the horizontal distance are related by:
    # x_sep = 2 * R * sin(phi), where R = H * c0 / (c0 * cos(theta_0))
    # Simpler: parameterize by the ray parameter p = sin(theta_0)/c(0).
    # At turning point: p = 1/c(z_t) => c(z_t) = 1/p => z_t = H*((1/p/c0)^2 - 1)

    # We find p by matching the horizontal distance.
    # For circular-arc rays: x_sep = 2 * H / (p * c0) * sqrt(1 - (p*c0)^2)
    # => p * c0 = sin(theta_0)

    # Use numerical integration for generality.
    def horizontal_distance(p):
        """Compute horizontal distance for ray parameter p."""
        z_turn = H * ((1.0 / (p * c0))**2 - 1.0)
        if z_turn <= 0:
            return 0.0
        z_arr = np.linspace(0, z_turn, 1000)
        c_z = c0 * np.sqrt(1 + z_arr / H)
        # dx/dz = p * c / sqrt(1 - p^2 * c^2)
        arg = 1.0 - (p * c_z)**2
        arg = np.clip(arg, 1e-12, None)
        integrand = p * c_z / np.sqrt(arg)
        return 2.0 * np.trapz(integrand, z_arr)

    def travel_time_calc(p):
        """Compute travel time for ray parameter p."""
        z_turn = H * ((1.0 / (p * c0))**2 - 1.0)
        if z_turn <= 0:
            return 0.0
        z_arr = np.linspace(0, z_turn, 1000)
        c_z = c0 * np.sqrt(1 + z_arr / H)
        arg = 1.0 - (p * c_z)**2
        arg = np.clip(arg, 1e-12, None)
        integrand = 1.0 / (c_z * np.sqrt(arg))
        return 2.0 * np.trapz(integrand, z_arr)

    # Find p that gives the desired x_sep.
    from scipy.optimize import brentq
    p_max = 0.999 / c0  # Nearly vertical.
    p_min = 1.0 / (c0 * np.sqrt(1 + 200.0 / H))  # Deep ray.

    try:
        p_opt = brentq(lambda p: horizontal_distance(p) - x_sep, p_min, p_max,
                       xtol=1e-10)
    except ValueError:
        # If brentq fails, return a simple parabolic approximation.
        x_ray = np.linspace(0, x_sep, n_steps)
        z_ray = 4 * (x_sep / 4.0) * (x_ray / x_sep) * (1 - x_ray / x_sep)
        tt = x_sep / c0 * 1e3  # Rough estimate [s].
        return x_ray, z_ray, tt

    z_turn = H * ((1.0 / (p_opt * c0))**2 - 1.0)
    tt = travel_time_calc(p_opt)

    # Build ray path.
    z_half = np.linspace(0, z_turn, n_steps // 2)
    c_z = c0 * np.sqrt(1 + z_half / H)
    arg = 1.0 - (p_opt * c_z)**2
    arg = np.clip(arg, 1e-12, None)
    integrand = p_opt * c_z / np.sqrt(arg)

    # Cumulative horizontal distance.
    x_half = np.zeros_like(z_half)
    for i in range(1, len(z_half)):
        x_half[i] = x_half[i - 1] + (integrand[i] + integrand[i - 1]) / 2 * (
            z_half[i] - z_half[i - 1])

    # Full ray: down then up (mirror).
    x_ray = np.concatenate([x_half, x_sep - x_half[::-1]])
    z_ray = np.concatenate([z_half, z_half[::-1]])

    # Convert travel time to seconds.
    tt_seconds = tt * 1e3  # Mm/s -> travel time in ks, convert to s.

    return x_ray, z_ray, tt_seconds


def compute_travel_time_from_point(x_s, z_s, x_r, z_r):
    """Estimate travel time between two arbitrary interior points using straight-line average.

    This is a simplified estimate: tau ~ distance / c_avg, where c_avg is the
    average sound speed along the straight line between the two points.

    Args:
        x_s: Source x [Mm].
        z_s: Source z (depth) [Mm].
        x_r: Receiver x [Mm].
        z_r: Receiver z (depth) [Mm].

    Returns:
        Estimated travel time [s].
    """
    dist = np.sqrt((x_r - x_s)**2 + (z_r - z_s)**2)
    # Average depth along straight line.
    z_avg = (z_s + z_r) / 2.0
    c_avg = sound_speed(np.maximum(z_avg, 0))
    if c_avg == 0:
        return np.inf
    return dist / c_avg  # [Mm / (Mm/s)] = [s] ... but c is in Mm/s already.


# --- Compute ray paths for different separations ---
separations = [20, 40, 60, 80]  # [Mm]
ray_paths = {}

for sep in separations:
    x_r, z_r, tt = compute_ray_path(sep)
    ray_paths[sep] = (x_r, z_r, tt)

# --- Compute Fresnel zone kernel for one separation ---
sep_main = 40.0  # [Mm]
x_ray_main, z_ray_main, tt_total = ray_paths[sep_main]

# Source and receiver positions.
x1, z1 = 0.0, 0.0       # Source at surface.
x2, z2 = sep_main, 0.0   # Receiver at surface.

# 2D grid for kernel computation.
nx_grid = 200
nz_grid = 100
x_grid_1d = np.linspace(-5, sep_main + 5, nx_grid)
z_grid_1d = np.linspace(0, 30, nz_grid)  # Depth [Mm].
x_grid, z_grid = np.meshgrid(x_grid_1d, z_grid_1d)

# Travel times from source to each grid point, and from each grid point to receiver.
tau_1 = np.zeros_like(x_grid)
tau_2 = np.zeros_like(x_grid)

for i in range(nz_grid):
    for j in range(nx_grid):
        tau_1[i, j] = compute_travel_time_from_point(x1, z1, x_grid[i, j], z_grid[i, j])
        tau_2[i, j] = compute_travel_time_from_point(x_grid[i, j], z_grid[i, j], x2, z2)

# Direct travel time (source to receiver along the ray).
tau_direct = tt_total

# Detour time: Delta_tau = tau_1 + tau_2 - tau_direct.
delta_tau = tau_1 + tau_2 - tau_direct

# Fresnel zone kernel (based on Eq. 65 of the paper):
# K(r) ~ sin(omega_0 * Delta_tau) / (tau_1 * tau_2) * exp(-(a * Delta_tau / (T0/2))^2)
omega_0 = 2 * np.pi * 3e-3  # Central frequency: 3 mHz [rad/s].
T0 = 1.0 / (3e-3)           # Period at 3 mHz [s].
a_damp = 2.0                 # Damping parameter (controls Fresnel zone width).

# Avoid division by zero.
tau_1_safe = np.where(tau_1 > 0, tau_1, 1e-10)
tau_2_safe = np.where(tau_2 > 0, tau_2, 1e-10)

K_fresnel = (np.sin(omega_0 * delta_tau) / (tau_1_safe * tau_2_safe)
             * np.exp(-(a_damp * delta_tau / (T0 / 2))**2))

# Mask out regions far from the ray (where kernel is negligible).
K_fresnel = np.where(np.abs(delta_tau) < 2 * T0, K_fresnel, 0)

# --- Plot results ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (1) Ray paths for different separations.
ax = axes[0, 0]
colors_ray = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
for idx, sep in enumerate(separations):
    xr, zr, tt = ray_paths[sep]
    ax.plot(xr, zr, color=colors_ray[idx], lw=2,
            label=f"$\\Delta$ = {sep} Mm, $\\tau$ = {tt:.0f} s")
ax.set_xlabel("Horizontal Distance [Mm] / 수평 거리")
ax.set_ylabel("Depth [Mm] / 깊이")
ax.set_title("Acoustic Ray Paths / 음향 ray 경로")
ax.invert_yaxis()
ax.legend(fontsize=10)
ax.set_aspect("equal")

# (2) Fresnel zone kernel — 2D cross-section.
ax = axes[0, 1]
vmax_k = np.percentile(np.abs(K_fresnel), 98)
im = ax.pcolormesh(x_grid_1d, z_grid_1d, K_fresnel,
                   shading="auto", cmap="RdBu_r", vmin=-vmax_k, vmax=vmax_k)
ax.plot(x_ray_main, z_ray_main, "k-", lw=2, label="Ray path")
ax.plot([x1, x2], [z1, z2], "k*", ms=15)
ax.set_xlabel("Horizontal Distance [Mm] / 수평 거리")
ax.set_ylabel("Depth [Mm] / 깊이")
ax.set_title(f"Fresnel Zone Kernel ($\\Delta$ = {sep_main:.0f} Mm)\n"
             "\"Banana-Doughnut\" 민감도 커널")
ax.invert_yaxis()
ax.legend(fontsize=10)
plt.colorbar(im, ax=ax, label="$K(\\mathbf{r})$")

# (3) Horizontal cross-sections at different depths.
ax = axes[1, 0]
depth_slices = [2, 5, 10, 15]  # [Mm]
for d in depth_slices:
    z_idx = np.argmin(np.abs(z_grid_1d - d))
    ax.plot(x_grid_1d, K_fresnel[z_idx, :], lw=1.5, label=f"z = {d} Mm")
ax.set_xlabel("Horizontal Distance [Mm] / 수평 거리")
ax.set_ylabel("$K(x, z)$")
ax.set_title("Kernel Cross-Sections at Different Depths\n다양한 깊이에서의 커널 단면")
ax.legend(fontsize=10)
ax.axhline(0, color="gray", ls=":", lw=0.8)

# (4) Vertical cross-section at midpoint — comparing Ray vs Fresnel.
ax = axes[1, 1]
x_mid = sep_main / 2.0
x_mid_idx = np.argmin(np.abs(x_grid_1d - x_mid))

# Fresnel kernel profile.
k_profile = K_fresnel[:, x_mid_idx]

# Ray kernel: delta function at ray depth at midpoint.
z_ray_at_mid = np.interp(x_mid, x_ray_main, z_ray_main)
ray_kernel = np.exp(-0.5 * ((z_grid_1d - z_ray_at_mid) / 0.3)**2)
ray_kernel = ray_kernel / np.max(ray_kernel) * np.max(np.abs(k_profile))

ax.plot(z_grid_1d, k_profile, "b-", lw=2, label="Fresnel zone kernel")
ax.plot(z_grid_1d, ray_kernel, "r--", lw=2, label="Ray kernel (Gaussian approx.)")
ax.axvline(z_ray_at_mid, color="gray", ls=":", lw=1, label=f"Ray depth = {z_ray_at_mid:.1f} Mm")
ax.set_xlabel("Depth [Mm] / 깊이")
ax.set_ylabel("$K(z)$ at midpoint / 중간점 커널")
ax.set_title("Vertical Profile: Ray vs Fresnel at Midpoint\n수직 프로파일: 중간점에서 Ray vs Fresnel")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Key observation: The Fresnel zone kernel shows near-zero sensitivity ON the ray")
print(f"path, with maximum sensitivity at the first Fresnel zone — the 'banana-doughnut'")
print(f"structure predicted by finite-wavelength theory (Paper §6).")

---
## 요약 / Summary

이 notebook에서는 Gizon & Birch (2005)의 국소 일진학 핵심 기법 5가지를 수치적으로 구현했습니다.

This notebook numerically implemented 5 key techniques from Gizon & Birch (2005) on Local Helioseismology.

| Part | 구현 내용 / Implementation | 논문 섹션 / Paper Section | 핵심 결과 / Key Result |
|------|---------------------------|---------------------------|------------------------|
| 1 | l-ν diagram (p-mode & f-mode 분산 관계) | §2 | 25개 p-mode ridge와 f-mode의 분산 관계 시각화 / Visualized dispersion relations of 25 p-mode ridges and f-mode |
| 2 | Ring-diagram 분석 | §4 | Lorentzian ring 생성, 유동에 의한 ring shift 검출 및 복원 / Created Lorentzian rings, detected and recovered flow-induced ring shifts |
| 3 | Time-distance cross-covariance | §5 | 1D 파동 시뮬레이션으로 time-distance diagram 재현 / Reproduced time-distance diagram via 1D wave simulation |
| 4 | Travel time & flow detection | §5.2–5.3 | τ_diff ∝ U 선형 관계 확인 / Confirmed linear relationship τ_diff ∝ U |
| 5 | Sensitivity kernels (Ray vs Fresnel) | §6 | "Banana-doughnut" 커널 구조 재현, ray 위 민감도 ≈ 0 확인 / Reproduced "banana-doughnut" kernel structure, confirmed near-zero sensitivity on ray |

### 다음 논문 / Next Papers

이 논문에서 다룬 기법들은 다음 논문들에서 더 발전됩니다:

The techniques covered in this paper are further developed in the following papers:

- **Christensen-Dalsgaard (2002)** — 일진학의 역사와 전체적 개관 / Helioseismology history and overview
- **Kosovichev (1996)** — Time-distance inversion의 원조 / Pioneer of time-distance inversion
- **Birch & Gizon (2007)** — Born 근사 sensitivity kernel / Born-approximation sensitivity kernels
- **Zhao et al. (2012)** — SDO/HMI를 이용한 현대 time-distance 분석 / Modern time-distance analysis with SDO/HMI